In [ ]:
# Importing all necessary libraries

# ipython magic functions for automatic reloading code changes in .py files
%load_ext autoreload
%autoreload 2

# internal packages
import os
# external packages
import torch
import torchvision
# import torchvision
import numpy as np
import sklearn
from matplotlib import pyplot as plt
from sklearn.preprocessing import scale
# from sklearn.neighbors import NearestNeighbors
# from scipy.stats import mode

%matplotlib inline

# specify base paths

base_path = "/export/share/peters57dm/Verbund/deepsync"
model_name = "autoencoder.pth"

if not os.path.exists(base_path):
    os.mkdir(base_path)

print("Versions")
print("torch: ",torch.__version__)
print("torchvision: ",torchvision.__version__)
print("numpy: ", np.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
import sys
sys.path.append('/export/share/peters57dm/Verbund/deepsync/helper.py')
from helper import (
    Autoencoder,
    assign_unified_labels,
    load_example,
    load_usps,
    load_htru,
    load_pendigits,
    load_optdigits,
    load_mnist,
    load_letterrecognition,
    load_gaussian_blobs,
    calculate_squared_differences_vectorized,
    find_local_core_points_fast,
    plot_and_save,
    plot_and_save_two_lists,
    get_high_conf_labels,
    encode_batchwise,
    plot_2d_dataset,
    detect_device,
    get_train_and_testloader,
    load_pretrained_model,
    pretrain_model,
    calc_distance_threshold,
    distance_nearest_neighbor
    )

# Train/Load Pretrain Model

In [ ]:
# path to were we want to save/load the model to/from
pretrained_model_name = "pretrained_" + model_name
pretrained_model_path = os.path.join(base_path, pretrained_model_name)

In [ ]:
def print_accuracies(gt_labels, epoch_labels, return_results=True):
  from sklearn.metrics import adjusted_mutual_info_score as ami
  from sklearn.metrics import adjusted_rand_score as ari
  labeled_pts_mask = epoch_labels > -1
  lbld_pts_score = ami(gt_labels[labeled_pts_mask],
                                epoch_labels[labeled_pts_mask])
  all_pts_score = ami(gt_labels, epoch_labels)
  lbld_pts_ari = ari(gt_labels[labeled_pts_mask],
                     epoch_labels[labeled_pts_mask])
  all_pts_ari = ari (gt_labels, epoch_labels)
  print("Labeled pts AMI = ", lbld_pts_score)
  print("All pts AMI = ", all_pts_score)
  print("labeled pts ARI = ", lbld_pts_ari)
  print("All pts ARI = ", all_pts_ari)
  
  if return_results:
    return lbld_pts_score, all_pts_score, lbld_pts_ari, all_pts_ari

# Deep sync with GT labels

In [ ]:
# using mahalanobis distance for assignment
def assign_unlabeled_points(data, labels):
    from scipy.spatial.distance import mahalanobis
    unique_labels = set(labels) - {-1}  # Get unique cluster labels excluding -1

    # Compute cluster centroids, covariance matrices, and thresholds
    centroids = {}
    cov_matrices = {}
    inv_cov_matrices = {}
    thresholds = {}

    outlier_points = data[labels == -1]
    if len(outlier_points) == 0: # all points are labeled
        return labels # return the same labels

    for label in unique_labels:
        cluster_points = data[labels == label]
        centroids[label] = np.mean(cluster_points, axis=0)
        cov_matrices[label] = np.cov(cluster_points, rowvar=False)

        # Compute inverse covariance matrix (handle singular matrices)
        try:
            inv_cov_matrices[label] = np.linalg.inv(cov_matrices[label])
        except np.linalg.LinAlgError:
            inv_cov_matrices[label] = np.linalg.pinv(cov_matrices[label])  # Use pseudo-inverse if singular

        # Compute the Mahalanobis distance for each point in the cluster to its centroid
        distances = [mahalanobis(p, centroids[label], inv_cov_matrices[label]) for p in outlier_points]
        thresholds[label] = np.percentile(distances, 10)
        # distances = [mahalanobis(p, centroids[label], inv_cov_matrices[label]) for p in cluster_points]
        # thresholds[label] = np.max(distances) * 1.05

    # Assign unlabeled points (-1) to the nearest centroid based on Mahalanobis distance
    for i, point in enumerate(data):
        if labels[i] == -1:
            min_distance = float('inf')
            assigned_label = -1

            for label in unique_labels:
                distance = mahalanobis(point, centroids[label], inv_cov_matrices[label])
                if distance < min_distance:
                    min_distance = distance
                    assigned_label = label

            # Assign only if within the calculated threshold for that cluster
            if assigned_label != -1 and min_distance <= thresholds[assigned_label]:
                labels[i] = assigned_label

    return labels

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.stats import mode

def check_equality_in_n_consequtive_cols(prediction_matrix, n, crop):
    cols_to_check = prediction_matrix
    if not crop is None:
        cols_to_check = prediction_matrix[:, crop-n:crop]
    i = 0
    all_equal = cols_to_check[:, i] == cols_to_check[:, i+1]
    while i+1 < cols_to_check.shape[1]:
        check = cols_to_check[:, i] == cols_to_check[:, i+1]
        all_equal = np.logical_and(all_equal, check)
        i += 1
    custom_labels_mask = prediction_matrix[:, i] < 0
    all_equal[custom_labels_mask] = False
    return all_equal

In [ ]:
#deep_sync
deep_sync_model_name = "deep_sync_" + model_name
deep_sync_model_path = os.path.join(base_path, deep_sync_model_name)
deep_sync_path = os.path.join(base_path, "deep_sync.pth")

def deep_sync_model(device, model, data, dataset_name, trainloader,
                    testloader, optimizer, n_check, k, percent, training_iterations = 20):
   # Create folder to save results in
   directory = f'/export/share/peters57dm/Verbund/deepsync/results/mahalanobis_and_rc_att_rep/{dataset_name}'
   if not os.path.exists(directory):
      os.makedirs(directory)

   embedded, gt_labels = encode_batchwise(testloader, model, device)
   labels_over_iterations = np.zeros((len(data), training_iterations + 1)) - 10
   # squared_diffs = calculate_squared_differences_vectorized(data)
   # core_points_mask = find_core_points(squared_diffs, k)
   core_points_mask, _ = find_local_core_points_fast(embedded, k, percent)
   plot_2d_dataset(embedded, gt_labels.numpy(),
                  core_points_mask=core_points_mask,
                  centers=None, fixed_scales = False, save=None)
   losses_tracker = {'total':[], 'repel':[], 'attract':[]}
   eval_tracker = {'ari_labeled':[],'ari_total':[],'ami_labeled':[],'ami_total':[]}
   original_labels = torch.zeros_like(gt_labels) - 1
   original_labels[np.where(np.diag(core_points_mask)==1)[0]] = gt_labels[np.where(np.diag(core_points_mask)==1)[0]]

   labels_over_iterations[:,0] = original_labels
   i = 0
   break_flag = False
   while(i < training_iterations): 
      for batch, batch_labels, ids in trainloader:
         
         batch_data = batch.to(device)
         embedded = model.encode(batch_data)
         
         iteration_labels = labels_over_iterations[:, i][ids]
         attract_loss = 0
         repel_loss = 0
         unique_labels = np.unique(iteration_labels)
         unique_labels = unique_labels[unique_labels > -1]
         
         for l in unique_labels:
            # calculate attraction loss
            label_mask = iteration_labels == l
            label_pts = embedded[label_mask]
            label_square_diffs = (label_pts.unsqueeze(0) - label_pts.unsqueeze(1)).pow(2).sum(2)
            label_weights = 1 - (label_square_diffs / torch.max(label_square_diffs))
            n_label_pts = len(label_pts)
            _att_val = 1/n_label_pts**2 * torch.exp(-label_square_diffs*label_weights).sum()
            if torch.isnan(_att_val) :
               print(f"Attraction loss is nan at label {l}")
               continue
            attract_loss += 1 - _att_val

            # calculate repeling loss
            other_pts_mask = np.logical_and(iteration_labels != l, iteration_labels > -1)
            other_pts = embedded[other_pts_mask]
            other_pts_square_diffs = (label_pts.unsqueeze(0) - other_pts.unsqueeze(1)).pow(2).sum(2)
            label_weights = 1 # fixed label weight
            _rep_val = 1/len(other_pts_square_diffs)**2 * torch.exp(-other_pts_square_diffs * label_weights).sum()
            if torch.isnan(_rep_val) :
               print(f"Repel loss is nan at label {l}")
               continue
            repel_loss += _rep_val
            # repel_loss += (1/(len(other_pts_square_diffs)**2)) * (other_pts_square_diffs * label_weights).sum(0).sum()

         # local_rc_loss = 1 - r_c
         if len(unique_labels) == 0:
            print("No labels in this batch")
            break_flag = True
            continue
         loss = 1/len(unique_labels) * (attract_loss + repel_loss)
         # print(r_c)
         # Backward pass
         optimizer.zero_grad()
         loss.backward()
         optimizer.step()
      
      if break_flag:
         print("Done training.")
         break
      losses_tracker['total'].append(loss.detach().numpy())
      losses_tracker["repel"].append(repel_loss.detach().numpy())
      losses_tracker['attract'].append(attract_loss.detach().numpy())

      print("Attraction loss = ", attract_loss)
      print("Repeling loss = ", repel_loss)
      print("total loss = ", loss)
      train_embedded_data, _ = encode_batchwise(testloader, model, device)
      
      # Mahalanobis label assignment
      current_epoch_labels = labels_over_iterations[:,i]
      current_epoch_labels = assign_unlabeled_points(
         train_embedded_data, current_epoch_labels)

      if i > n_check:
         crop = i
      else :
         crop = None

   #   crop = i > n_check
      high_confidence_labels = check_equality_in_n_consequtive_cols(
         labels_over_iterations, n_check, crop)
      previous_epoch_labels = labels_over_iterations[:,i]
      unified_labels = assign_unified_labels(previous_epoch_labels,
                                             current_epoch_labels)
      unified_labels[high_confidence_labels] = previous_epoch_labels[high_confidence_labels]

      # 4 - Store the labels
      labels_over_iterations[:, i+1] = unified_labels
      # print(f"In epoch {i+1}, eps = {eps}")
      ami_l, ami_t, ari_l, ari_t = print_accuracies(gt_labels, unified_labels, return_results=True)
      eval_tracker['ami_labeled'].append(ami_l)
      eval_tracker['ami_total'].append(ami_t)
      eval_tracker['ari_labeled'].append(ari_l)
      eval_tracker['ari_total'].append(ari_t)

      # Only for plotting and nmi calculation
      if i % 1 == 0:
         # plot_2d_dataset(train_embedded_data, train_true_labels,
         #                 core_points_mask=high_confidence_labels,
         #                 centers=None, fixed_scales = False, save=None)
         saving_path = os.path.join(directory, '{0:03}'.format(i) + '.jpg')
         plot_2d_dataset(train_embedded_data, unified_labels,
                         centers=None, fixed_scales = False, save=saving_path)
         # print(r_c)
      i += 1
      print(np.all(high_confidence_labels))
      print(high_confidence_labels)
      if np.all(high_confidence_labels):
         print(f"apply early stopping after {i} iterations, all points are labeled confidently.")
         break
   print("Training finished")
   # Create plot to see finish result
   embedded_data, lbls = encode_batchwise(testloader, model, device)
   # # Plots
   plot_2d_dataset(embedded_data, gt_labels, fixed_scales=False)
   plot_and_save(losses_tracker["attract"],
                  title='Attraction Loss',
                  xlabel="Epochs",
                  ylabel="Value",
                  save_path=os.path.join(directory, 'attraction_loss.jpg'))
   plot_and_save(losses_tracker["repel"],
                  title='Repelling Loss',
                  xlabel="Epochs",
                  ylabel="Value",
                  save_path=os.path.join(directory, 'repelling_loss.jpg'))
   plot_and_save(losses_tracker["total"],
                  title='Total Sync Loss',
                  xlabel="Epochs",
                  ylabel="Value",
                  save_path=os.path.join(directory, 'sync_loss.jpg'))
   plot_and_save_two_lists(
       eval_tracker['ami_labeled'], eval_tracker['ami_total'],
       title="Adjusted Mutual Information",
       xlabel='Epochs',
       ylabel="Value",
       legend_label1='AMI Labeled',
       legend_label2="AMI All",
       save_path=os.path.join(directory, 'AMI.jpg')
    )
   plot_and_save_two_lists(
       eval_tracker['ari_labeled'], eval_tracker['ari_total'],
       title="Adjusted Rand Index",
       xlabel='Epochs',
       ylabel="Value",
       legend_label1='ARI Labeled',
       legend_label2="ARI All",
       save_path=os.path.join(directory, 'ARI.jpg')
    )
   # save model
   # torch.save(model.state_dict(), deep_sync_model_path)
   return model, labels_over_iterations

# Execution

In [ ]:
# Choose data, set parameters
batch_size = 512
# data, gt_labels, data_name, normalize = load_two_moons()
# data, gt_labels, data_name, normalize = load_gaussian_blobs()
# data, gt_labels, data_name, normalize = load_runEx()
# data, gt_labels, data_name, normalize = load_example()
# data = torch.cat((data[gt_labels == 2], data[gt_labels == 3]),0)
# gt_labels = torch.cat((gt_labels[gt_labels == 2] , gt_labels[gt_labels == 3]),0)
# data, gt_labels, data_name, normalize = load_htru()
# data, gt_labels, data_name, normalize = load_two_circles()
# data, gt_labels, data_name, normalize = load_mnist()
# data, gt_labels, data_name, normalize = load_fmnist()
data, gt_labels, data_name, normalize = load_usps()
# data, gt_labels, data_name, normalize = load_kmnist()
# data, gt_labels, data_name, normalize = load_pendigits()
# data, gt_labels, data_name, normalize = load_optdigits()
# data, gt_labels, data_name, normalize = load_banknotes()
# data, gt_labels, data_name, normalize = load_wafer()
# data, gt_labels, data_name, normalize = load_iris()
# data, gt_labels, data_name, normalize = load_motestrain()
# data, gt_labels, data_name, normalize = load_letterrecognition()
# data, gt_labels, data_name, normalize = load_mini_mnist()


# normalize = None
# normalize = 1
# normalize = 0
if normalize is not None:
    data = torch.from_numpy(scale(data, axis=normalize)).float()

# core points
k = 50
percent = 0.1
# squared_diffs = calculate_squared_differences_vectorized(data)
# core_points, th = find_local_core_points_fast(data, k, percent)
# core_points = find_core_points(squared_diffs, k)
n_check = 3

gt_n_clusters = len(np.unique(gt_labels))

print("Data Set Information")
print("Number of data points: ", data.shape[0])
print("Number of dimensions: ", data.shape[1])
print("Normalize: ", normalize)
print(f"Mean: {data.mean():.2f}, Standard deviation: {data.std():.2f}")
print(f"Min: {data.min():.2f}, Max: {data.max():.2f}")
print("Number of classes: ", gt_n_clusters)
gt_uniques = np.unique(gt_labels, return_counts=True)
print("Class distribution:\n", dict(zip(gt_uniques[0], gt_uniques[1])))


do_pretrain = True
do_deep_sync_train = True

embedded_space_dim = min(data.shape[1], 10)
# The size of the mini-batch that is passed in each trainings iteration
# define loss function
loss_fn = torch.nn.MSELoss()


# training_iterations
pretrain_training_iterations = 10
clustering_training_iterations = 50

# the size eps for the neighbourhood is defined by the mean distance across data points of a batch and will be scaled by eps_scale
eps_scale = 0.3
# Number of neighbours, if False, then eps method is chosen

# Set device on which the model should be trained on (For most of you this will be the CPU)
device = detect_device()
print("Use device: ", device)
print("Dimensionality of embedded space: ", embedded_space_dim)
print("Batch size: ", batch_size)
print("eps-scale: ", eps_scale)
print("Number of neighbours: ", k)
print("Pretrain iterations: {0} , Clustering iterations: {1} ".format(pretrain_training_iterations, clustering_training_iterations))

core_points = None
plot_2d_dataset(data.numpy(), gt_labels.numpy(), core_points)

# DeepSync Training

In [ ]:
model = Autoencoder(input_dim=data.shape[1], embedding_size=embedded_space_dim)
# train and testloader
trainloader, testloader = get_train_and_testloader(data, gt_labels, batch_size)

#### The learning rate specifies the step size of the gradient descent algorithm
learning_rate_pretrain = 1e-3
optimizer_pretrain = torch.optim.Adam(model.parameters(), lr = learning_rate_pretrain)

# Pretrain
print("=====> Start Pretrain <=====")
if do_pretrain:
    model = pretrain_model(device, pretrained_model_path,
                           model, data, trainloader,
                           optimizer_pretrain, loss_fn,
                           pretrain_training_iterations)
else:
    model = load_pretrained_model(model)
# define optimizer
learning_rate = 0.0001
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

print("=====> Start deep_sync <=====")
if do_deep_sync_train:
    model, labels_over_iterations = deep_sync_model(device, model, data, data_name,
                                                    trainloader, testloader, optimizer,
                                                    n_check, k, percent,
                                                    clustering_training_iterations)